In [1]:
import os
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from google import genai
from google.genai import types
from typing import List, Tuple

In [2]:
# Configure Gemini
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [3]:
def calculate_cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
    """Calculate cosine similarity between two vectors"""
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    
    # Manual calculation
    dot_product = np.dot(vec1, vec2)
    magnitude1 = np.linalg.norm(vec1)
    magnitude2 = np.linalg.norm(vec2)
    
    if magnitude1 == 0 or magnitude2 == 0:
        return 0.0
    
    return dot_product / (magnitude1 * magnitude2)

In [4]:
def calculate_euclidean_distance(vec1: List[float], vec2: List[float]) -> float:
    """Calculate Euclidean distance between two vectors"""
    return np.linalg.norm(np.array(vec1) - np.array(vec2))

In [5]:
def find_most_similar(
    query_embedding: List[float], 
    embeddings: List[List[float]], 
    texts: List[str],
    method: str = "cosine"
) -> List[Tuple[str, float]]:
    """Find most similar texts to query"""
    similarities = []
    
    for i, embedding in enumerate(embeddings):
        if method == "cosine":
            similarity = calculate_cosine_similarity(query_embedding, embedding)
        else:  # euclidean
            # Convert distance to similarity (inverse)
            distance = calculate_euclidean_distance(query_embedding, embedding)
            similarity = 1 / (1 + distance)  # Convert to 0-1 range
        
        similarities.append((texts[i], similarity))
    
    # Sort by similarity (descending)
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities

In [6]:
# Example: How to create embeddings for RAG applications
# These documents would typically come from your knowledge base
documents = [
    "The cat sat on the mat",
    "A feline rested on the rug",
    "The dog played in the park",
    "Machine learning is fascinating",
    "Deep learning is a subset of ML"
]


In [7]:
def get_embedding(
    text: str,
    model_name: str = "gemini-embedding-001",
    task_type: str = "RETRIEVAL_DOCUMENT",  # "RETRIEVAL_QUERY" for queries
) -> list[float]:
    """Get embedding for a single text using Gemini"""
    result = client.models.embed_content(
        model=model_name,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type),
    )
    assert result.embeddings and result.embeddings[0].values is not None
    return list(result.embeddings[0].values)

In [8]:
def get_embeddings_batch(
    texts: list[str],
    model_name: str = "gemini-embedding-001",
    task_type: str = "RETRIEVAL_DOCUMENT",
) -> list[list[float]]:
    """Get embeddings for multiple texts in a single request"""
    result = client.models.embed_content(
        model=model_name,
        contents=texts,
        config=types.EmbedContentConfig(task_type=task_type),
    )
    assert result.embeddings is not None
    return [list(e.values) for e in result.embeddings if e.values is not None]

In [9]:
embeddings = get_embeddings_batch(documents)

In [10]:
# User query for retrieval augmented generation
query = "A kitten was sleeping on the carpet"
query_embedding = get_embedding(query, task_type="RETRIEVAL_QUERY")

In [11]:
results = find_most_similar(query_embedding, embeddings, documents, method="cosine")
for text, score in results:
    print(f"{score:.4f}  {text}")

0.7809  A feline rested on the rug
0.7180  The cat sat on the mat
0.6351  The dog played in the park
0.6040  Machine learning is fascinating
0.5889  Deep learning is a subset of ML


In [12]:
euclidean_results = find_most_similar(query_embedding, embeddings, documents, method="euclidean")

In [13]:
print("\nEuclidean Distance Results:")
for text, score in euclidean_results[:3]:
    print(f"  {score:.4f}: {text}")


Euclidean Distance Results:
  0.6017: A feline rested on the rug
  0.5711: The cat sat on the mat
  0.5393: The dog played in the park
